# OCR-only Current Pipeline Starter

Notebook này tạo `submission.csv` bằng pipeline OCR-only hiện tại: OCR profile `resize_960_contrast_1.35_sharpen`, product extractor train từ `train_labels.csv`, variant rules + OCR evidence override. Không dùng old `product_name` từ submission cũ.

## Cell 1 - Install Dependencies

In [ ]:
!pip install rapidocr onnxruntime tqdm scikit-learn lightgbm rapidfuzz pyyaml -q

## Cell 2 - Load Data

In [ ]:
import csv, re, sys, time, zipfile
from pathlib import Path

import pandas as pd

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

try:
    from PIL import Image, ImageEnhance, ImageFilter
except ImportError as exc:
    raise RuntimeError('Pillow is required for OCR preprocessing') from exc

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

IMAGE_DIR_CANDIDATES = (
    'test_images/images', 'test_images', 'images', 'test/images',
    'test/test_images/images', 'test/test_images',
)
IMAGE_ZIP_NAMES = ('test_images.zip', 'images.zip', 'test/test_images.zip', 'test/images.zip')

def input_roots():
    roots = []
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        comp_root = kaggle_input / 'competitions'
        if comp_root.exists():
            roots.extend(sorted(comp_root.iterdir()))
        roots.extend(sorted(kaggle_input.iterdir()))
    for local in [Path('data/raw'), Path('public_dataset'), Path('.')]:
        if local.exists():
            roots.append(local.resolve())
    out, seen = [], set()
    for root in roots:
        if root not in seen:
            seen.add(root)
            out.append(root)
    return out

def resolve_input_dir():
    for root in input_roots():
        if (root / 'test.csv').exists():
            return root
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for test_csv in sorted(kaggle_input.rglob('test.csv')):
            return test_csv.parent
    raise FileNotFoundError('Cannot find test.csv')

def find_images_dir(input_dir):
    for rel in IMAGE_DIR_CANDIDATES:
        images_dir = input_dir / rel
        if images_dir.is_dir() and any(images_dir.glob('*.jpg')):
            return images_dir
    return None

def find_images_zip(input_dir):
    for rel in IMAGE_ZIP_NAMES:
        zip_path = input_dir / rel
        if zip_path.is_file():
            return zip_path
    for zip_path in sorted(input_dir.rglob('*.zip')):
        name = zip_path.name.lower()
        if 'train' in name and 'test' not in name:
            continue
        if name in ('test_images.zip', 'images.zip') or name.endswith('images.zip'):
            return zip_path
    return None

def setup_images_dir(input_dir, work_dir):
    images_dir = find_images_dir(input_dir)
    if images_dir is not None:
        return images_dir
    zip_path = find_images_zip(input_dir)
    if zip_path is None:
        raise FileNotFoundError(f'No test images found under {input_dir}')
    extract_root = work_dir / 'dataset_images'
    images_dir = extract_root / 'images'
    if not any(images_dir.glob('*.jpg')):
        extract_root.mkdir(parents=True, exist_ok=True)
        print(f'Extracting {zip_path} ...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_root)
        for rel in ('test_images/images', 'test_images', 'images'):
            alt = extract_root / rel
            if alt.is_dir() and any(alt.glob('*.jpg')):
                return alt
    return images_dir

INPUT_DIR = resolve_input_dir()
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('reports/notebook_current_pipeline')
WORK_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR = setup_images_dir(INPUT_DIR, WORK_DIR)

TEST_CSV = INPUT_DIR / 'test.csv'
TRAIN_LABELS_CANDIDATES = [INPUT_DIR / 'train_labels.csv', Path('data/raw/train_labels.csv'), Path('public_dataset/train_labels.csv')]
OUTPUT_CSV = WORK_DIR / 'submission.csv'
OCR_CACHE_CSV = WORK_DIR / 'ocr_tmp.csv'
TMP_PRODUCT_CSV = WORK_DIR / 'tmp_product_extract.csv'

test_df = pd.read_csv(TEST_CSV, keep_default_na=False)
train_labels_df = None
for labels_path in TRAIN_LABELS_CANDIDATES:
    if labels_path.exists():
        train_labels_df = pd.read_csv(labels_path, keep_default_na=False)
        break
if train_labels_df is None:
    raise FileNotFoundError('train_labels.csv is required for this OCR-only product pipeline')

print(f'Input dir   : {INPUT_DIR}')
print(f'Images dir  : {IMAGES_DIR}')
print(f'Test rows   : {len(test_df):,}')
print(f'Train rows  : {len(train_labels_df):,}')
test_df.head(3)

## Cell 3 - Import Current Pipeline Logic

In [ ]:
from model_dispatcher import HybridProductExtractor, MISSING_PRODUCT, normalize_match_text
from scripts.generate_product_variants import (
    HALONG_NAN_TMP_PRODUCTS,
    evidence_product_from_ocr,
    evidence_cotden_product_from_ocr,
    evidence_override_product_from_ocr,
    rule_product_from_ocr,
)

print('Loaded current OCR-only product pipeline helpers')

## Cell 4 - OCR Engine

In [ ]:
from rapidocr import RapidOCR
import numpy as np

OCR_MAX_DIM = 960
OCR_CONTRAST = 1.35
OCR_SCORE_THRESHOLD = 0.35

ocr_engine = RapidOCR()
print('RapidOCR loaded | profile: resize_960_contrast_1.35_sharpen')

def load_image(image_id):
    path = IMAGES_DIR / f'{image_id}.jpg'
    if not path.exists():
        return None
    try:
        return Image.open(path).convert('RGB')
    except Exception:
        return None

def preprocess_image(img):
    w, h = img.size
    if max(w, h) > OCR_MAX_DIM:
        ratio = OCR_MAX_DIM / max(w, h)
        img = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
    img = ImageEnhance.Contrast(img).enhance(OCR_CONTRAST)
    return img.filter(ImageFilter.SHARPEN)

def postprocess_ocr(text):
    text = re.sub(r'\s+', ' ', str(text)).strip()
    tokens = text.split()
    if not tokens:
        return ''
    deduped = [tokens[0]]
    for tok in tokens[1:]:
        if tok.lower() != deduped[-1].lower():
            deduped.append(tok)
    return ' '.join(deduped)

def run_ocr_text(image_id):
    img = load_image(image_id)
    if img is None:
        return ''
    img_np = np.array(preprocess_image(img))
    try:
        result = ocr_engine(img_np)
        txts = list(getattr(result, 'txts', None) or [])
        scores = list(getattr(result, 'scores', None) or [1.0] * len(txts))
        boxes = getattr(result, 'boxes', None)
        rows = []
        for idx, txt in enumerate(txts):
            score = float(scores[idx]) if idx < len(scores) else 1.0
            if score <= OCR_SCORE_THRESHOLD:
                continue
            y_key = x_key = idx
            if boxes is not None and idx < len(boxes):
                box = boxes[idx]
                y_key = min(p[1] for p in box)
                x_key = min(p[0] for p in box)
            rows.append((y_key, x_key, str(txt).strip()))
        rows.sort(key=lambda item: (item[0], item[1]))
        return postprocess_ocr(' '.join(txt for _, _, txt in rows if txt))
    except Exception:
        return ''

first_id = test_df['image_id'].iloc[0]
print(first_id, run_ocr_text(first_id)[:120])

## Cell 5 - Build OCR Cache

In [ ]:
SAVE_EVERY = 50

rows = []
start = time.time()
for i, image_id in enumerate(tqdm(test_df['image_id'].astype(str).tolist(), desc='OCR test'), start=1):
    rows.append({'image_id': image_id, 'ocr_text': run_ocr_text(image_id), 'product_name': MISSING_PRODUCT})
    if i % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_csv(OCR_CACHE_CSV, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)

ocr_submission = pd.DataFrame(rows, columns=['image_id', 'ocr_text', 'product_name'])
ocr_submission.to_csv(OCR_CACHE_CSV, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
print(f'OCR cache saved: {OCR_CACHE_CSV} | rows={len(ocr_submission):,} | minutes={(time.time()-start)/60:.1f}')
ocr_submission.head(3)

## Cell 6 - Product Extract From OCR Only

In [ ]:
train_product = train_labels_df[['image_id', 'ocr_text', 'product_name']].copy()
train_product['ocr_text'] = train_product['ocr_text'].fillna('').astype(str)
train_product['product_name'] = train_product['product_name'].fillna('').astype(str).str.strip()

extractor = HybridProductExtractor(random_state=42).fit(train_product)
tmp_product = ocr_submission[['image_id', 'ocr_text']].copy()
tmp_product['product_name'] = [extractor.predict(text) for text in tqdm(tmp_product['ocr_text'], desc='Product extract')]
tmp_product.to_csv(TMP_PRODUCT_CSV, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
print(f'TMP product saved: {TMP_PRODUCT_CSV}')
tmp_product['product_name'].astype(str).str.strip().value_counts().head(20)

## Cell 7 - Apply Current Variant Rules

In [ ]:
def is_nonempty(value):
    return str(value).strip() != ''

def fill_empty(df, fills):
    out = df.copy()
    fill_values = pd.Series(fills, index=out.index).fillna('').astype(str)
    old_empty = ~out['product_name'].map(is_nonempty)
    fill_nonempty = fill_values.str.strip().ne('')
    out.loc[old_empty & fill_nonempty, 'product_name'] = fill_values[old_empty & fill_nonempty]
    out['product_name'] = out['product_name'].replace('', MISSING_PRODUCT)
    return out[['image_id', 'ocr_text', 'product_name']]

# Base is intentionally blank: no old product_name is used.
base = ocr_submission[['image_id', 'ocr_text']].copy()
base['product_name'] = ''

tmp_products = tmp_product['product_name'].fillna('').astype(str).str.strip()
halong_nan_mask = tmp_products.map(lambda value: normalize_match_text(value) in HALONG_NAN_TMP_PRODUCTS)
work = fill_empty(base, tmp_products.where(halong_nan_mask, ''))

rule_fills = work['ocr_text'].map(rule_product_from_ocr)
work = fill_empty(work, rule_fills)

evidence_fills = work['ocr_text'].map(evidence_product_from_ocr)
work = fill_empty(work, evidence_fills)

cotden_all = work['ocr_text'].map(lambda text: evidence_cotden_product_from_ocr(text, strict=False))
work = fill_empty(work, cotden_all)

work['product_name'] = [
    evidence_override_product_from_ocr(text, product)
    for text, product in zip(work['ocr_text'].tolist(), work['product_name'].tolist())
]
work['product_name'] = work['product_name'].replace('', MISSING_PRODUCT)

submission = work[['image_id', 'ocr_text', 'product_name']].copy()
print('empty product:', int(submission['product_name'].astype(str).str.strip().eq('').sum()))
submission['product_name'].astype(str).str.strip().value_counts().head(20)

## Cell 8 - Validate And Export

In [ ]:
required_cols = ['image_id', 'ocr_text', 'product_name']
assert list(submission.columns) == required_cols
assert len(submission) == len(test_df)
assert submission['image_id'].astype(str).tolist() == test_df['image_id'].astype(str).tolist()

submission.to_csv(OUTPUT_CSV, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
print(f'Wrote {OUTPUT_CSV}')
submission.head(10)

## Notes

- Pipeline này không đọc bất kỳ file old-product submission nào.
- Nếu muốn tái hiện đúng file local `ocr_only_baseline_variants_v2/...ocr_override.csv`, hãy chạy cùng source code repo hiện tại và cùng OCR backend/profile.